
 05 - Rotten Tomatoes Filmes - Camada Prata
 MVP-3 — Engenharia de Dados

 Este notebook transforma os dados de filmes do Rotten Tomatoes
 da camada Bronze para a camada Prata.

 Achados do profiling
 - 17.712 registros;
 - nenhuma duplicidade;
 - nenhuma chave rotten_tomatoes_link nula;
 - 17.712 chaves distintas;
 - existência de valores nulos em atributos descritivos;
 - datas armazenadas originalmente como texto.

 Transformações
 - preservação da chave natural rotten_tomatoes_link;
 - conversão controlada de campos de data;
 - conversão controlada de campos numéricos;
 - preservação dos valores originais relevantes;
 - criação de metadados de processamento;
 - validações de qualidade;
 - persistência em formato Delta.

In [0]:
from pyspark.sql import functions as F


CATALOGO = "mvp3_eng_dados"

SCHEMA_BRONZE = "bronze"
SCHEMA_PRATA = "prata"


TABELA_BRONZE = (
    f"{CATALOGO}.{SCHEMA_BRONZE}.rt_filmes"
)

TABELA_PRATA = (
    f"{CATALOGO}.{SCHEMA_PRATA}.rt_filmes"
)


print("✓ Configuração carregada")
print(f"Origem:  {TABELA_BRONZE}")
print(f"Destino: {TABELA_PRATA}")

✓ Configuração carregada
Origem:  mvp3_eng_dados.bronze.rt_filmes
Destino: mvp3_eng_dados.prata.rt_filmes


In [0]:
df_rt_filmes_bronze = spark.table(
    TABELA_BRONZE
)


qtd_bronze = (
    df_rt_filmes_bronze.count()
)


print(
    f"✓ RT Filmes Bronze carregado: "
    f"{qtd_bronze:,} registros"
)

df_rt_filmes_bronze.printSchema()

✓ RT Filmes Bronze carregado: 17,712 registros
root
 |-- rotten_tomatoes_link: string (nullable = true)
 |-- movie_title: string (nullable = true)
 |-- movie_info: string (nullable = true)
 |-- critics_consensus: string (nullable = true)
 |-- content_rating: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- directors: string (nullable = true)
 |-- authors: string (nullable = true)
 |-- actors: string (nullable = true)
 |-- original_release_date: string (nullable = true)
 |-- streaming_release_date: string (nullable = true)
 |-- runtime: string (nullable = true)
 |-- production_company: string (nullable = true)
 |-- tomatometer_status: string (nullable = true)
 |-- tomatometer_rating: string (nullable = true)
 |-- tomatometer_count: string (nullable = true)
 |-- audience_status: string (nullable = true)
 |-- audience_rating: string (nullable = true)
 |-- audience_count: string (nullable = true)
 |-- tomatometer_top_critics_count: string (nullable = true)
 |-- tomatomet

In [0]:
print("Colunas disponíveis na Bronze:\n")

for coluna in df_rt_filmes_bronze.columns:
    print(coluna)

Colunas disponíveis na Bronze:

rotten_tomatoes_link
movie_title
movie_info
critics_consensus
content_rating
genres
directors
authors
actors
original_release_date
streaming_release_date
runtime
production_company
tomatometer_status
tomatometer_rating
tomatometer_count
audience_status
audience_rating
audience_count
tomatometer_top_critics_count
tomatometer_fresh_critics_count
tomatometer_rotten_critics_count
_fonte
_arquivo_origem
_data_ingestao


In [0]:
df_rt_filmes_tratado = (
    df_rt_filmes_bronze

    .withColumn(
        "data_lancamento_original",
        F.to_date(
            F.col("original_release_date"),
            "yyyy-MM-dd"
        )
    )

    .withColumn(
        "data_lancamento_streaming",
        F.to_date(
            F.col("streaming_release_date"),
            "yyyy-MM-dd"
        )
    )
)


print("✓ Campos de data convertidos")

✓ Campos de data convertidos


In [0]:
df_rt_filmes_tratado = (
    df_rt_filmes_tratado

    .withColumn(
        "duracao_minutos",
        F.col("runtime").cast("integer")
    )

    .withColumn(
        "avaliacao_tomatometer",
        F.col("tomatometer_rating").cast("double")
    )

    .withColumn(
        "quantidade_tomatometer",
        F.col("tomatometer_count").cast("long")
    )

    .withColumn(
        "avaliacao_audiencia",
        F.col("audience_rating").cast("double")
    )

    .withColumn(
        "quantidade_audiencia",
        F.col("audience_count").cast("long")
    )
)


print("✓ Campos numéricos convertidos")

✓ Campos numéricos convertidos


In [0]:
df_rt_filmes_tratado = (
    df_rt_filmes_tratado

    .withColumn(
        "id_filme",
        F.sha2(
            F.col("rotten_tomatoes_link"),
            256
        )
    )

    .withColumn(
        "fonte_dados",
        F.lit("Rotten Tomatoes")
    )

    .withColumn(
        "data_hora_processamento",
        F.current_timestamp()
    )
)


print("✓ Identificador e metadados criados")

✓ Identificador e metadados criados


In [0]:
colunas_tecnicas_bronze = [
    "_fonte",
    "_arquivo_origem",
    "_data_ingestao"
]


df_rt_filmes_prata = (
    df_rt_filmes_tratado
    .drop(
        *colunas_tecnicas_bronze
    )
)


print("✓ Estrutura Prata preparada")

✓ Estrutura Prata preparada


In [0]:
colunas_prioritarias = [
    "id_filme",
    "rotten_tomatoes_link",
    "movie_title",
    "data_lancamento_original",
    "data_lancamento_streaming",
    "duracao_minutos",
    "avaliacao_tomatometer",
    "quantidade_tomatometer",
    "avaliacao_audiencia",
    "quantidade_audiencia"
]


colunas_restantes = [
    coluna
    for coluna in df_rt_filmes_prata.columns
    if coluna not in colunas_prioritarias
]


df_rt_filmes_prata = (
    df_rt_filmes_prata
    .select(
        *colunas_prioritarias,
        *colunas_restantes
    )
)


df_rt_filmes_prata.printSchema()

root
 |-- id_filme: string (nullable = true)
 |-- rotten_tomatoes_link: string (nullable = true)
 |-- movie_title: string (nullable = true)
 |-- data_lancamento_original: date (nullable = true)
 |-- data_lancamento_streaming: date (nullable = true)
 |-- duracao_minutos: integer (nullable = true)
 |-- avaliacao_tomatometer: double (nullable = true)
 |-- quantidade_tomatometer: long (nullable = true)
 |-- avaliacao_audiencia: double (nullable = true)
 |-- quantidade_audiencia: long (nullable = true)
 |-- movie_info: string (nullable = true)
 |-- critics_consensus: string (nullable = true)
 |-- content_rating: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- directors: string (nullable = true)
 |-- authors: string (nullable = true)
 |-- actors: string (nullable = true)
 |-- original_release_date: string (nullable = true)
 |-- streaming_release_date: string (nullable = true)
 |-- runtime: string (nullable = true)
 |-- production_company: string (nullable = true)
 |-- tom

In [0]:
qtd_registros = (
    df_rt_filmes_prata.count()
)


qtd_chaves_nulas = (
    df_rt_filmes_prata
    .filter(
        F.col(
            "rotten_tomatoes_link"
        ).isNull()
    )
    .count()
)


qtd_chaves_distintas = (
    df_rt_filmes_prata
    .select(
        "rotten_tomatoes_link"
    )
    .distinct()
    .count()
)


print(
    f"Registros: "
    f"{qtd_registros:,}"
)

print(
    f"Chaves nulas: "
    f"{qtd_chaves_nulas:,}"
)

print(
    f"Chaves distintas: "
    f"{qtd_chaves_distintas:,}"
)

Registros: 17,712
Chaves nulas: 0
Chaves distintas: 17,712


In [0]:
qtd_ids_nulos = (
    df_rt_filmes_prata
    .filter(
        F.col("id_filme").isNull()
    )
    .count()
)


qtd_ids_distintos = (
    df_rt_filmes_prata
    .select("id_filme")
    .distinct()
    .count()
)


print(
    f"IDs nulos: "
    f"{qtd_ids_nulos:,}"
)

print(
    f"IDs distintos: "
    f"{qtd_ids_distintos:,}"
)

IDs nulos: 0
IDs distintos: 17,712


In [0]:
datas_invalidas_original = (
    df_rt_filmes_prata
    .filter(
        F.col("original_release_date").isNotNull()
        &
        F.col("data_lancamento_original").isNull()
    )
    .count()
)


datas_invalidas_streaming = (
    df_rt_filmes_prata
    .filter(
        F.col("streaming_release_date").isNotNull()
        &
        F.col("data_lancamento_streaming").isNull()
    )
    .count()
)


print(
    "Datas originais não convertidas: "
    f"{datas_invalidas_original:,}"
)

print(
    "Datas de streaming não convertidas: "
    f"{datas_invalidas_streaming:,}"
)

Datas originais não convertidas: 0
Datas de streaming não convertidas: 0


In [0]:
campos_numericos = [
    (
        "runtime",
        "duracao_minutos"
    ),
    (
        "tomatometer_rating",
        "avaliacao_tomatometer"
    ),
    (
        "tomatometer_count",
        "quantidade_tomatometer"
    ),
    (
        "audience_rating",
        "avaliacao_audiencia"
    ),
    (
        "audience_count",
        "quantidade_audiencia"
    )
]


for original, convertido in campos_numericos:

    invalidos = (
        df_rt_filmes_prata
        .filter(
            F.col(original).isNotNull()
            &
            F.col(convertido).isNull()
        )
        .count()
    )

    print(
        f"{original}: "
        f"{invalidos:,} valores não convertidos"
    )

runtime: 0 valores não convertidos
tomatometer_rating: 0 valores não convertidos
tomatometer_count: 0 valores não convertidos
audience_rating: 0 valores não convertidos
audience_count: 0 valores não convertidos


In [0]:
assert qtd_registros == qtd_bronze, (
    "Quantidade de registros foi alterada."
)

assert qtd_chaves_nulas == 0, (
    "Existem rotten_tomatoes_link nulos."
)

assert qtd_chaves_distintas == qtd_registros, (
    "rotten_tomatoes_link não é único."
)

assert qtd_ids_nulos == 0, (
    "Existem id_filme nulos."
)

assert qtd_ids_distintos == qtd_registros, (
    "id_filme não é único."
)


print("✓ Validações obrigatórias aprovadas")

✓ Validações obrigatórias aprovadas


In [0]:
display(
    df_rt_filmes_prata
    .select(
        "id_filme",
        "movie_title",
        "rotten_tomatoes_link",
        "data_lancamento_original",
        "duracao_minutos",
        "avaliacao_tomatometer",
        "avaliacao_audiencia"
    )
    .limit(10)
)

id_filme,movie_title,rotten_tomatoes_link,data_lancamento_original,duracao_minutos,avaliacao_tomatometer,avaliacao_audiencia
4bd83058befe7a98e4d3eedf19feeac7fc3eee6892ff6ce1ba5719210ec38776,Percy Jackson & the Olympians: The Lightning Thief,m/0814255,2010-02-12,119,49.0,53.0
309053ff882903caabf48cc11e1a52ff23605ebb2f76cd4a89539cd90bf7ff64,Please Give,m/0878835,2010-04-30,90,87.0,64.0
93e49c50855217e93a81cca4b7dd3b38206b9e124923302d663e7ab137c55270,10,m/10,1979-10-05,122,67.0,53.0
a57f6a39c5dc87ffd3d460a573dc1e445753c26b6c04f567ca168873c39983bf,12 Angry Men (Twelve Angry Men),m/1000013-12_angry_men,1957-04-13,95,100.0,97.0
e3043dcc9ec976c953af76bb2c30f8a28b55718b6eb07c45d5aaa566f2097268,"20,000 Leagues Under The Sea",m/1000079-20000_leagues_under_the_sea,1954-01-01,127,89.0,74.0
b2b5fe273c56be168233836a921aa9c15fd713c346d46cd8c91af08ef1fcb0ce,"10,000 B.C.",m/10000_bc,2008-03-07,109,8.0,37.0
16bff4ed5fd5a55f049b0eaab1431ba6339bb2b6702e4a47dbd4a0b7e5984d85,The 39 Steps,m/1000121-39_steps,1935-08-01,80,96.0,86.0
2b39bd5f7250c2f499f71e73008ece03fe8baf74bba0046d96312a33d08bc74d,3:10 to Yuma,m/1000123-310_to_yuma,1957-08-07,92,96.0,79.0
08044c9c740c50abcf481b30c6adaecf040777062af3093c0101b87bfb84f82a,Charly (A Heartbeat Away),m/10002008-charly,2002-09-27,103,20.0,87.0
896b0daf98226eeeb4162c81a2ded8455e7ca011f546843cba3819383c3a2560,Abraham Lincoln,m/1000204-abraham_lincoln,1930-11-08,97,82.0,40.0


In [0]:
(
    df_rt_filmes_prata.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        TABELA_PRATA
    )
)


print(
    f"✓ Tabela criada: "
    f"{TABELA_PRATA}"
)

✓ Tabela criada: mvp3_eng_dados.prata.rt_filmes


In [0]:
assert spark.catalog.tableExists(
    TABELA_PRATA
), "Tabela Prata não encontrada."


df_rt_filmes_validacao = (
    spark.table(
        TABELA_PRATA
    )
)


qtd_prata = (
    df_rt_filmes_validacao.count()
)


print(
    f"Bronze: "
    f"{qtd_bronze:,}"
)

print(
    f"Prata:  "
    f"{qtd_prata:,}"
)


assert qtd_bronze == qtd_prata, (
    "Bronze e Prata possuem quantidades diferentes."
)


print("✓ Quantidade de registros preservada")

Bronze: 17,712
Prata:  17,712
✓ Quantidade de registros preservada


In [0]:
print("=" * 60)
print("ROTTEN TOMATOES FILMES - CAMADA PRATA CONCLUÍDA")
print("=" * 60)


print("\nOrigem:")
print(TABELA_BRONZE)


print("\nDestino:")
print(TABELA_PRATA)


print("\nResultado:")

print(
    f"✓ Registros Bronze: "
    f"{qtd_bronze:,}"
)

print(
    f"✓ Registros Prata: "
    f"{qtd_prata:,}"
)


print("\nTransformações:")

print("✓ Chave natural preservada")
print("✓ Identificador técnico criado")
print("✓ Datas convertidas")
print("✓ Campos numéricos tipados")
print("✓ Valores originais preservados")
print("✓ Nulos legítimos preservados")
print("✓ Validações de qualidade executadas")


print("\nStatus:")
print("✓ RT FILMES PRATA VALIDADA")


print("\nPróxima etapa:")
print("06_rt_avaliacoes_prata")

ROTTEN TOMATOES FILMES - CAMADA PRATA CONCLUÍDA

Origem:
mvp3_eng_dados.bronze.rt_filmes

Destino:
mvp3_eng_dados.prata.rt_filmes

Resultado:
✓ Registros Bronze: 17,712
✓ Registros Prata: 17,712

Transformações:
✓ Chave natural preservada
✓ Identificador técnico criado
✓ Datas convertidas
✓ Campos numéricos tipados
✓ Valores originais preservados
✓ Nulos legítimos preservados
✓ Validações de qualidade executadas

Status:
✓ RT FILMES PRATA VALIDADA

Próxima etapa:
06_rt_avaliacoes_prata
